# Notebook 04 — Soil Moisture Data Processing

This notebook imports the raw weather dataset, extracts and cleans soil moisture profiles (Surface Wetness `Soil_Moisture_Top` and Root Zone Wetness `Soil_Moisture_Root`), and exports a clean dataset for predictive modeling.

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Processed Data Folder:", config.PROCESSED_DIR)

Processed Data Folder: D:\A1\coco\Coconut_Yield_Prediction\processed


## 2. Load Raw Weather and Moisture Data

In [2]:
raw_weather_path = config.RAW_DIR / "weather_raw.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(f"{raw_weather_path.name} not found. Run Notebook 02 first.")

df_raw = pd.read_csv(raw_weather_path)
print("Loaded raw weather data shape:", df_raw.shape)
print(df_raw[["District", "Year", "Month", "Soil_Moisture_Top", "Soil_Moisture_Root"]].head())

Loaded raw weather data shape: (728, 11)
    District  Year  Month  Soil_Moisture_Top  Soil_Moisture_Root
0  Makandura  2019      1               0.60                0.60
1  Makandura  2019      2               0.50                0.52
2  Makandura  2019      3               0.40                0.45
3  Makandura  2019      4               0.42                0.45
4  Makandura  2019      5               0.47                0.49


## 3. Clean and Transform Soil Moisture

We verify that the moisture ratios are bounded within [0.0, 1.0]. We also calculate the soil moisture profile difference (Root Zone Moisture minus Top Surface Moisture) to represent groundwater storage profiles.

In [3]:
df_soil = df_raw[["District", "Year", "Month", "Soil_Moisture_Top", "Soil_Moisture_Root"]].copy()

# Bounding check
df_soil["Soil_Moisture_Top"] = df_soil["Soil_Moisture_Top"].clip(lower=0.0, upper=1.0)
df_soil["Soil_Moisture_Root"] = df_soil["Soil_Moisture_Root"].clip(lower=0.0, upper=1.0)

# Calculate the difference (wetness deficit/surplus)
df_soil["Soil_Moisture_Diff"] = df_soil["Soil_Moisture_Root"] - df_soil["Soil_Moisture_Top"]

# Check for missing values
print("Missing values summary:\n", df_soil.isnull().sum())

# Handle missing values if any exist by grouping by district/month
if df_soil.isnull().sum().sum() > 0:
    medians = df_soil.groupby(["District", "Month"])[["Soil_Moisture_Top", "Soil_Moisture_Root", "Soil_Moisture_Diff"]].transform("median")
    df_soil = df_soil.fillna(medians)

print(df_soil.describe())

Missing values summary:
 District              0
Year                  0
Month                 0
Soil_Moisture_Top     0
Soil_Moisture_Root    0
Soil_Moisture_Diff    0
dtype: int64
              Year      Month  Soil_Moisture_Top  Soil_Moisture_Root  \
count   728.000000  728.00000         728.000000          728.000000   
mean   2022.000000    7.00000           0.694121            0.687912   
std       2.001375    3.74423           0.130849            0.130656   
min    2019.000000    1.00000           0.360000            0.450000   
25%    2020.000000    4.00000           0.600000            0.590000   
50%    2022.000000    7.00000           0.700000            0.680000   
75%    2024.000000   10.00000           0.790000            0.780000   
max    2025.000000   13.00000           0.950000            0.990000   

       Soil_Moisture_Diff  
count          728.000000  
mean            -0.006209  
std              0.024605  
min             -0.050000  
25%             -0.020000  
5

## 4. Export Clean Soil Moisture Dataset

In [4]:
out_path = config.PROCESSED_DIR / "soil_moisture_clean.csv"
df_soil.to_csv(out_path, index=False)
print(f"[SUCCESS] Cleaned soil moisture dataset saved to {out_path.name}")

[SUCCESS] Cleaned soil moisture dataset saved to soil_moisture_clean.csv
